# LeetCode 1410 — HTML Entity Parser bằng Python

Link đề: https://leetcode.com/problems/html-entity-parser/ · Độ khó: **Medium** · Chủ đề: Hash Table, String

Skeleton LeetCode:

```python
class Solution:
    def entityParser(self, text: str) -> str:
        ...
```

## 1. Mục tiêu học

1. Hiểu bài toán **thay thế chuỗi không chồng lấn, quét một lượt từ trái sang phải** và vì sao "thay lần lượt từng mẫu" dễ sai.
2. Thấy rõ lỗi **giải mã hai lần** khi `replace("&amp;", "&")` chạy **trước** — và cách sửa bằng thứ tự (để `&amp;` cuối cùng).
3. Cài đặt quét một lượt với `dict` (tra thực thể bắt đầu tại mỗi `&`), dùng `str.startswith(prefix, i)`.
4. Dùng `re.sub` với **nhóm lựa chọn** và **hàm thay thế**, hiểu tính chất "không chồng lấn, trái sang phải" của regex.
5. Tránh lỗi: dùng `html.unescape` (giải mã quá tay), cắt cửa sổ quá ngắn (quên `&frasl;` dài 7), nhảy qua `&` lồng nhau, regex tham lam `&.*;`.

## 2. Đề bài & ví dụ

Bộ phân tích thực thể HTML nhận một đoạn văn bản và thay **các thực thể** bằng ký tự tương ứng. Chỉ có 6 thực thể:

| Thực thể | Ký tự |
|---|---|
| `&quot;` | `"` |
| `&apos;` | `'` |
| `&amp;` | `&` |
| `&gt;` | `>` |
| `&lt;` | `<` |
| `&frasl;` | `/` |

Trả về văn bản sau khi thay. Mọi chuỗi khác (ví dụ `&ambassador;`) giữ nguyên.

**Ví dụ 1:** `"&amp; is an HTML entity but &ambassador; is not."` → `"& is an HTML entity but &ambassador; is not."`

**Ví dụ 2:** `"and I quote: &quot;...&quot;"` → `"and I quote: \"...\""`

**Ràng buộc:** `1 <= text.length <= 10^5`; chuỗi có thể chứa bất kỳ ký tự nào trong 256 ký tự ASCII.

**Điểm tinh tế:** mỗi thực thể chỉ được thay **một lần**, không "giải mã lại" kết quả. Ví dụ `"&amp;gt;"` → `"&gt;"` (không phải `">"`): sau khi `&amp;` thành `&`, chuỗi `&gt;` vừa xuất hiện **không** được thay tiếp. (Ví dụ này tự dựng theo đúng định nghĩa của đề — trình chấm LeetCode cũng cho kết quả `"&gt;"`; nên đối chiếu lại đề gốc nếu cần.)

In [1]:
import html
import random
import re
import time
from typing import List

ENTITIES = {"&quot;": '"', "&apos;": "'", "&amp;": "&", "&gt;": ">", "&lt;": "<", "&frasl;": "/"}

examples = [
    ("&amp; is an HTML entity but &ambassador; is not.", "& is an HTML entity but &ambassador; is not."),
    ("and I quote: &quot;...&quot;", 'and I quote: "..."'),
]
for t, want in examples:
    print(repr(t), "->", repr(want))

'&amp; is an HTML entity but &ambassador; is not.' -> '& is an HTML entity but &ambassador; is not.'
'and I quote: &quot;...&quot;' -> 'and I quote: "..."'


## 3. Kiến thức nền

### 3.1 `dict` tra cứu thực thể

`dict` cho tra `O(1)` trung bình: `ENTITIES.get(s)` trả ký tự hoặc `None`.

In [2]:
print(ENTITIES.get("&gt;"), ENTITIES.get("&ambassador;"), "&frasl;" in ENTITIES)
print("độ dài các thực thể:", sorted({len(e) for e in ENTITIES}))

> None True
độ dài các thực thể: [4, 5, 6, 7]


### 3.2 `str.replace` thay **mọi** lần xuất hiện, quét trái → phải, không chồng lấn

`s.replace(old, new)` trả chuỗi mới, `O(len(s))`. Nhiều lời gọi nối tiếp nhau thì lời gọi sau **nhìn thấy kết quả** của lời gọi trước — nguồn gốc của lỗi thứ tự.

In [3]:
s = "&amp;gt;"
print(s.replace("&amp;", "&"))
print(s.replace("&amp;", "&").replace("&gt;", ">"), "<- lời gọi thứ hai thay luôn &gt; vừa sinh ra")

&gt;
> <- lời gọi thứ hai thay luôn &gt; vừa sinh ra


### 3.3 `str.startswith(prefix, start)`

Kiểm tra chuỗi con bắt đầu tại vị trí `start` mà **không cần cắt chuỗi** (`text[i:i+k]` tạo chuỗi mới).

In [4]:
text = "a &lt; b"
print(text.startswith("&lt;", 2), text.startswith("&lt;", 0), text.startswith("&gt;", 2))

True False False


### 3.4 Ghép kết quả bằng list rồi `"".join`

Với chuỗi dài `10^5`, `res += ch` trong vòng lặp có thể chậm; gom vào list rồi `"".join(parts)` là `O(n)`.

In [5]:
parts = []
for ch in "abc":
    parts.append(ch.upper())
print("".join(parts))

ABC


### 3.5 Biểu thức chính quy: nhóm lựa chọn và hàm thay thế

- `re.escape(s)` thoát ký tự đặc biệt.
- `"|".join(...)` tạo nhóm lựa chọn `(a|b|c)`.
- `re.sub(pattern, func, text)`: với mỗi khớp `m`, thay bằng `func(m)`. Regex quét **trái sang phải, không chồng lấn** và **không quét lại** phần đã thay ⇒ đúng ngữ nghĩa "một lượt".

In [6]:
pat = re.compile("|".join(map(re.escape, ENTITIES)))
print(pat.pattern)
print(pat.sub(lambda m: ENTITIES[m.group()], "&amp;gt; &lt;b&gt;"))

\&quot;|\&apos;|\&amp;|\&gt;|\&lt;|\&frasl;
&gt; <b>


### 3.6 `html.unescape` của thư viện chuẩn — **không** dùng cho bài này

`html.unescape` giải mã **mọi** thực thể HTML5 (hàng nghìn thực thể, cả dạng số `&#65;`, và vài thực thể không cần dấu `;`). Đề chỉ cho phép 6 thực thể ⇒ dùng nó sẽ sai (Lỗi 2, mục 13).

In [7]:
for t in ["&amp;", "&nbsp;x", "&#65;", "&ltb"]:
    print(repr(t), "->", repr(html.unescape(t)))

'&amp;' -> '&'
'&nbsp;x' -> '\xa0x'
'&#65;' -> 'A'
'&ltb' -> '<b'


## 4. Ý tưởng nền tảng / trực giác

Tính tay `text = "x &amp;lt; y &frasl &ambassador; &&gt;"` bằng cách **đọc từ trái sang phải** như người:

1. `x ` — giữ nguyên.
2. Gặp `&` tại `&amp;lt;`: thử các thực thể ⇒ khớp `&amp;` ⇒ ghi `&`, **nhảy qua** 5 ký tự. Phần còn lại `lt; y …` đọc như văn bản thường (không quay lại ghép với `&` vừa ghi) ⇒ ra `&lt;`.
3. ` y ` giữ nguyên.
4. `&frasl` thiếu dấu `;` ⇒ không phải thực thể ⇒ ghi `&`, đi tiếp 1 ký tự.
5. `&ambassador;` ⇒ không khớp thực thể nào ⇒ ghi nguyên.
6. `&&gt;`: `&` đầu tiên không khớp (ký tự sau là `&`) ⇒ ghi `&`; `&` thứ hai khớp `&gt;` ⇒ ghi `>`.

Kết quả: `"x &lt; y &frasl &ambassador; &>"`.

Quy tắc rút ra: **tại mỗi vị trí `i`, nếu một thực thể bắt đầu tại `i` thì thay và nhảy qua nó, ngược lại chép một ký tự**. Không bao giờ quay lại.

In [8]:
text = "x &amp;lt; y &frasl &ambassador; &&gt;"
i, out = 0, []
while i < len(text):
    for ent, ch in ENTITIES.items():
        if text.startswith(ent, i):
            print(f"i={i:2d}: khớp {ent!r} -> {ch!r}")
            out.append(ch); i += len(ent)
            break
    else:
        out.append(text[i]); i += 1
print("kết quả:", repr("".join(out)))

i= 2: khớp '&amp;' -> '&'
i=34: khớp '&gt;' -> '>'
kết quả: 'x &lt; y &frasl &ambassador; &>'


## 5. Cách 1 — Brute Force / baseline: `replace` nối tiếp (và cái bẫy thứ tự)

### Ý tưởng
Gọi `replace` cho từng thực thể. Cách viết "ngây thơ" đặt `&amp;` **đầu tiên**:

In [9]:
def parse_replace_naive(text):
    for ent in ["&amp;", "&quot;", "&apos;", "&gt;", "&lt;", "&frasl;"]:   # &amp; đầu tiên: SAI
        text = text.replace(ent, ENTITIES[ent])
    return text

for t, want in examples:
    print(parse_replace_naive(t) == want)
print(repr(parse_replace_naive("&amp;gt;")), "<- sai: đúng ra phải là '&gt;'")

True
True
'>' <- sai: đúng ra phải là '&gt;'


Hai ví dụ của đề vẫn qua, nhưng `"&amp;gt;"` bị **giải mã hai lần**: `&amp;` → `&` tạo ra `&gt;` mới, rồi lời gọi kế tiếp biến nó thành `>`.

**Sửa:** thay `&amp;` **cuối cùng**. Các ký tự sinh ra bởi 5 thực thể kia (`" ' > < /`) không thể tạo thành thực thể mới (thực thể cần `&`), và sau khi thay `&amp;` thì không còn lời gọi nào nữa.

In [10]:
def parse_replace(text):
    for ent in ["&quot;", "&apos;", "&gt;", "&lt;", "&frasl;", "&amp;"]:   # &amp; CUỐI CÙNG
        text = text.replace(ent, ENTITIES[ent])
    return text

for t, want in examples + [("&amp;gt;", "&gt;"), ("&&gt;", "&>"), ("&amp;amp;", "&amp;")]:
    print(repr(t), "->", repr(parse_replace(t)), "| đúng?", parse_replace(t) == want)

'&amp; is an HTML entity but &ambassador; is not.' -> '& is an HTML entity but &ambassador; is not.' | đúng? True
'and I quote: &quot;...&quot;' -> 'and I quote: "..."' | đúng? True
'&amp;gt;' -> '&gt;' | đúng? True
'&&gt;' -> '&>' | đúng? True
'&amp;amp;' -> '&amp;' | đúng? True


### Độ phức tạp và nút thắt

- Time `O(6·n)` = `O(n)` — mỗi `replace` quét toàn chuỗi một lần (chạy trong C, rất nhanh). Space `O(n)` cho chuỗi mới mỗi lần.
- Nút thắt **không phải tốc độ** mà là **độ tin cậy**: tính đúng phụ thuộc vào một lập luận tinh tế về thứ tự. Thêm một thực thể mới (ví dụ thực thể sinh ra `&` hay ký tự chữ) có thể làm vỡ lập luận. Cách quét một lượt (Cách 2–4) đúng **theo cấu trúc**, không cần lập luận thứ tự.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** "giải mã" là phép biến đổi **một lượt, trái sang phải**: đầu ra của một lần thay **không bao giờ** là đầu vào của lần thay khác. Thay vì chạy nhiều lượt (`replace` nối tiếp), ta tự quét **một lần** với con trỏ `i`.

**Bất biến:** tại mọi thời điểm, `out` là kết quả giải mã đúng của `text[:i]`, và `i` luôn nằm ở ranh giới hợp lệ (không ở giữa một thực thể đã thay).

**Quy tắc "nếu… thì…":**
- Nếu `text[i] != '&'` ⇒ chép ký tự, `i += 1`.
- Nếu `text[i] == '&'` và có thực thể `e` với `text.startswith(e, i)` ⇒ ghi ký tự của `e`, `i += len(e)`.
- Ngược lại ⇒ chép `'&'`, `i += 1` (để `&` tiếp theo, nếu có, vẫn được xét — ví dụ `&&gt;`).

Vì các thực thể đều kết thúc bằng `;` và chỉ chứa chữ cái sau `&`, không có hai thực thể nào cùng khớp tại một vị trí (không cái nào là tiền tố của cái khác) ⇒ thứ tự thử không quan trọng.

## 7. Các cách tiếp theo

### Cách 2 — Quét một lượt với `dict` + `startswith`

**Từng đoạn:**
1. `i = 0`, `out = []`.
2. Nếu `text[i] == '&'`: duyệt 6 thực thể, cái nào `startswith(e, i)` thì ghi ký tự, nhảy `len(e)`.
3. Không khớp hoặc không phải `&`: chép `text[i]`, `i += 1`.
4. Trả `"".join(out)`.

In [11]:
def parse_scan(text):
    out = []
    i, n = 0, len(text)
    while i < n:
        if text[i] == "&":
            for ent, ch in ENTITIES.items():
                if text.startswith(ent, i):
                    out.append(ch)
                    i += len(ent)
                    break
            else:
                out.append("&")
                i += 1
        else:
            out.append(text[i])
            i += 1
    return "".join(out)

for t, want in examples + [("&amp;gt;", "&gt;"), ("&&gt;", "&>")]:
    print(repr(parse_scan(t)), "| đúng?", parse_scan(t) == want)

'& is an HTML entity but &ambassador; is not.' | đúng? True
'and I quote: "..."' | đúng? True
'&gt;' | đúng? True
'&>' | đúng? True


Biến thể: thay vì thử 6 thực thể, tìm dấu `;` gần nhất **trong cửa sổ 7 ký tự** (`&frasl;` dài nhất) rồi tra `dict` một lần. Phải cẩn thận: nếu giữa `&` và `;` có một `&` khác thì chỉ chép `&` hiện tại rồi đi tiếp (Lỗi 3).

In [12]:
class SolutionScan:
    def entityParser(self, text: str) -> str:
        table = {"&quot;": '"', "&apos;": "'", "&amp;": "&",
                 "&gt;": ">", "&lt;": "<", "&frasl;": "/"}
        out = []
        i, n = 0, len(text)
        while i < n:
            if text[i] == "&":
                j = text.find(";", i, i + 7)          # ';' trong cửa sổ tối đa 7 ký tự
                if j != -1 and text[i:j + 1] in table:
                    out.append(table[text[i:j + 1]])
                    i = j + 1
                    continue
            out.append(text[i])
            i += 1
        return "".join(out)

print([SolutionScan().entityParser(t) == w for t, w in examples], repr(SolutionScan().entityParser("&&gt;&amp;lt;")))

[True, True] '&>&lt;'


Ở bản này, nếu `text[i:j+1]` chứa một `&` khác (như `"&&gt;"` khi `i` ở `&` đầu) thì nó không có trong bảng ⇒ chỉ chép **một** `&` và tăng `i` lên 1 — `&` thứ hai sẽ được xét ở vòng sau. Time `O(n)` (mỗi vị trí xét cửa sổ hằng số), Space `O(n)`.

### Cách 3 — Biểu thức chính quy `re.sub`

Một dòng: mẫu `&(?:quot|apos|amp|gt|lt|frasl);` và hàm thay thế tra `dict`. `re.sub` quét trái → phải, không chồng lấn, không quét lại kết quả ⇒ đúng ngữ nghĩa.

In [13]:
ENT_RE = re.compile(r"&(?:quot|apos|amp|gt|lt|frasl);")

def parse_regex(text):
    return ENT_RE.sub(lambda m: ENTITIES[m.group()], text)

for t, want in examples + [("&amp;gt;", "&gt;"), ("&&gt;", "&>")]:
    print(repr(parse_regex(t)), "| đúng?", parse_regex(t) == want)

'& is an HTML entity but &ambassador; is not.' | đúng? True
'and I quote: "..."' | đúng? True
'&gt;' | đúng? True
'&>' | đúng? True


In [14]:
class SolutionRegex:
    def entityParser(self, text: str) -> str:
        table = {"&quot;": '"', "&apos;": "'", "&amp;": "&",
                 "&gt;": ">", "&lt;": "<", "&frasl;": "/"}
        return re.sub(r"&(?:quot|apos|amp|gt|lt|frasl);", lambda m: table[m.group()], text)

print([SolutionRegex().entityParser(t) == w for t, w in examples])

[True, True]


Time `O(n)` (máy regex thử 6 nhánh ngắn tại mỗi `&`), Space `O(n)`. Ngắn nhất, nhưng cần hiểu regex.

### Cách 4 — Đánh dấu `&` gần nhất, xử lý khi gặp `;` (theo gợi ý của đề)

Quét từng ký tự, chép vào `out`. Ghi nhớ **vị trí trong `out`** của dấu `&` gần nhất (`amp`). Khi gặp `;`: nếu `out[amp:] + ";"` là một thực thể ⇒ xoá đoạn đó khỏi `out`, ghi ký tự thay thế. Sau khi thay, đặt `amp = None` để ký tự `&` **vừa sinh ra** (từ `&amp;`) không bị coi là đầu thực thể mới.

In [15]:
def parse_marker(text):
    out = []
    amp = None                        # vị trí trong out của '&' gần nhất chưa "đóng"
    for ch in text:
        if ch == "&":
            amp = len(out)
            out.append(ch)
        elif ch == ";" and amp is not None and len(out) - amp <= 6:
            cand = "".join(out[amp:]) + ";"
            if cand in ENTITIES:
                del out[amp:]
                out.append(ENTITIES[cand])
            else:
                out.append(ch)
            amp = None
        else:
            out.append(ch)
    return "".join(out)

for t, want in examples + [("&amp;gt;", "&gt;"), ("&&gt;", "&>"), ("&amp;amp;", "&amp;")]:
    print(repr(parse_marker(t)), "| đúng?", parse_marker(t) == want)

'& is an HTML entity but &ambassador; is not.' | đúng? True
'and I quote: "..."' | đúng? True
'&gt;' | đúng? True
'&>' | đúng? True
'&amp;' | đúng? True


Time `O(n)` (mỗi `;` kiểm tra ≤ 7 ký tự), Space `O(n)`.

## 8. TRACE từng bước

Trace Cách 2 (quét với `startswith`) trên `text = "a&&lt;b&amp;gt;"`:

| i | text[i] | Thử thực thể | Ghi | i mới | out |
|---|---|---|---|---|---|
| 0 | a | — | a | 1 | a |
| 1 | & | không khớp (sau là `&`) | & | 2 | a& |
| 2 | & | `&lt;` khớp | < | 6 | a&< |
| 6 | b | — | b | 7 | a&<b |
| 7 | & | `&amp;` khớp | & | 12 | a&<b& |
| 12 | g | — | g | 13 | a&<b&g |
| 13 | t | — | t | 14 | a&<b&gt |
| 14 | ; | — | ; | 15 | a&<b&gt; |

In [16]:
def trace_scan(text):
    out = []
    i = 0
    while i < len(text):
        if text[i] == "&":
            hit = next((e for e in ENTITIES if text.startswith(e, i)), None)
            if hit:
                out.append(ENTITIES[hit])
                print(f"i={i:2d} '&': khớp {hit!r:9s} -> ghi {ENTITIES[hit]!r}, nhảy tới {i + len(hit):2d} | out={''.join(out)!r}")
                i += len(hit)
                continue
            print(f"i={i:2d} '&': không khớp thực thể nào -> chép '&'          | out={''.join(out) + '&'!r}")
        out.append(text[i])
        i += 1
    return "".join(out)

print("kết quả:", repr(trace_scan("a&&lt;b&amp;gt;")))

i= 1 '&': không khớp thực thể nào -> chép '&'          | out='a&'
i= 2 '&': khớp '&lt;'    -> ghi '<', nhảy tới  6 | out='a&<'
i= 7 '&': khớp '&amp;'   -> ghi '&', nhảy tới 12 | out='a&<b&'
kết quả: 'a&<b&gt;'


Trace Cách 4 (đánh dấu `&`) để thấy vai trò của `amp = None` sau khi thay `&amp;`:

In [17]:
def trace_marker(text):
    out, amp = [], None
    for k, ch in enumerate(text):
        note = ""
        if ch == "&":
            amp = len(out); out.append(ch); note = f"đánh dấu amp={amp}"
        elif ch == ";" and amp is not None and len(out) - amp <= 6:
            cand = "".join(out[amp:]) + ";"
            if cand in ENTITIES:
                del out[amp:]; out.append(ENTITIES[cand]); note = f"{cand!r} -> {ENTITIES[cand]!r}, amp=None"
            else:
                out.append(ch); note = f"{cand!r} không phải thực thể, amp=None"
            amp = None
        else:
            out.append(ch)
        print(f"k={k:2d} ch={ch!r}: out={''.join(out)!r:12s} {note}")
    return "".join(out)

print("kết quả:", repr(trace_marker("&amp;gt;&x;")))

k= 0 ch='&': out='&'          đánh dấu amp=0
k= 1 ch='a': out='&a'         
k= 2 ch='m': out='&am'        
k= 3 ch='p': out='&amp'       
k= 4 ch=';': out='&'          '&amp;' -> '&', amp=None
k= 5 ch='g': out='&g'         
k= 6 ch='t': out='&gt'        
k= 7 ch=';': out='&gt;'       
k= 8 ch='&': out='&gt;&'      đánh dấu amp=4
k= 9 ch='x': out='&gt;&x'     
k=10 ch=';': out='&gt;&x;'    '&x;' không phải thực thể, amp=None
kết quả: '&gt;&x;'


## 9. Code nộp LeetCode

In [18]:
class Solution:
    def entityParser(self, text: str) -> str:
        table = {"&quot;": '"', "&apos;": "'", "&amp;": "&",
                 "&gt;": ">", "&lt;": "<", "&frasl;": "/"}
        out = []
        i, n = 0, len(text)
        while i < n:
            if text[i] == "&":
                for ent, ch in table.items():
                    if text.startswith(ent, i):     # thực thể bắt đầu tại i
                        out.append(ch)
                        i += len(ent)               # nhảy qua, không giải mã lại
                        break
                else:
                    out.append("&")                 # '&' lẻ: chép, xét tiếp ký tự sau
                    i += 1
            else:
                out.append(text[i])
                i += 1
        return "".join(out)

for t, want in examples:
    assert Solution().entityParser(t) == want
print("Solution đúng với 2 ví dụ của đề")

Solution đúng với 2 ví dụ của đề


## 10. Vì sao thuật toán đúng

**Đặc tả.** Kết quả đúng là: chia `text` thành dãy các mảnh từ trái sang phải, mỗi mảnh hoặc là **một thực thể** (khớp tại vị trí bắt đầu của nó) hoặc là **một ký tự**, với quy tắc "tại mỗi vị trí, nếu có thực thể bắt đầu ở đó thì lấy thực thể"; rồi thay mỗi thực thể bằng ký tự của nó.

**Không mơ hồ.** Tại một vị trí, tối đa **một** thực thể khớp: các thực thể đều có dạng `&` + chữ cái + `;`, và không cái nào là tiền tố của cái khác (dấu `;` kết thúc) ⇒ thứ tự thử không quan trọng.

**Cách 2 / Solution.** Bất biến: `out` = kết quả của `text[:i]`, và `i` là ranh giới mảnh. Mỗi vòng đọc đúng một mảnh bắt đầu tại `i` ⇒ bất biến giữ. Kết thúc `i = n` ⇒ `out` là kết quả của cả chuỗi. Vì không bao giờ đọc lại `out`, ký tự `&` sinh từ `&amp;` không thể tham gia thực thể mới.

**Cách 1 (thứ tự đúng).** Năm `replace` đầu chỉ sinh ký tự trong `" ' > < /` — không có `&` ⇒ không tạo thực thể mới và không phá thực thể có sẵn (một thực thể chỉ chứa chữ cái giữa `&` và `;`, nên không thể "giao" với một thực thể khác). `&amp;` thay cuối ⇒ các `&` sinh ra không bị xử lý thêm. Kết quả trùng với cách quét một lượt.

**Cách 3.** `re.sub` theo định nghĩa thay các khớp **không chồng lấn từ trái sang phải** và tìm tiếp **sau** khớp vừa rồi trên chuỗi gốc ⇒ đúng đặc tả.

**Cách 4.** Mỗi khi gặp `;`, đoạn `out[amp:]` chính là văn bản gốc từ `&` gần nhất (chưa bị thay vì `amp` bị huỷ sau mỗi lần thay hoặc mỗi `;`), nên kiểm tra đúng mảnh "thực thể kết thúc tại đây". `&` gần nhất là đúng vì thực thể không chứa `&` bên trong.

## 11. Phân tích Time / Space

Gọi `n = len(text)` (≤ 10⁵), `E = 6` thực thể, độ dài tối đa `L = 7`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. `replace` nối tiếp (`&amp;` cuối) | `O(E·n)` | `O(n)` | nhanh (chạy trong C) nhưng dựa vào lập luận thứ tự |
| 2. Quét + `startswith` | `O(n·E·L)` = `O(n)` | `O(n)` | đúng theo cấu trúc |
| 3. `re.sub` | `O(n)` | `O(n)` | ngắn nhất |
| 4. Đánh dấu `&` | `O(n·L)` = `O(n)` | `O(n)` | theo gợi ý của đề |

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

Nhãn của edge case được tính tay theo đặc tả "một lượt, trái sang phải" và **đối chiếu chéo** giữa các cách.

In [19]:
ALL = {
    "replace": parse_replace,
    "scan": parse_scan,
    "scan_submit": lambda t: SolutionScan().entityParser(t),
    "regex": parse_regex,
    "regex_submit": lambda t: SolutionRegex().entityParser(t),
    "marker": parse_marker,
    "submit": lambda t: Solution().entityParser(t),
}
edge = [
    ("&amp;gt;", "&gt;"),                 # không giải mã hai lần
    ("&amp;amp;", "&amp;"),
    ("&&gt;", "&>"),                      # & lẻ đứng trước thực thể
    ("&gt", "&gt"),                       # thiếu ;
    ("&GT;", "&GT;"),                     # phân biệt hoa thường
    ("&frasl;&frasl;", "//"),             # thực thể dài nhất
    ("&quot;&apos;&lt;&gt;", "\"'<>"),
    ("&;", "&;"), ("&", "&"), (";", ";"),
    ("&&&&", "&&&&"),
    ("x &amp;lt; y &frasl &ambassador; &&gt;", "x &lt; y &frasl &ambassador; &>"),
    ("&lt;&amp;quot;&gt;", "<&quot;>"),
    ("&fras;l", "&fras;l"),
    ("abc", "abc"),
]
for name, f in ALL.items():
    for t, want in examples + edge:
        assert f(t) == want, (name, t, f(t), want)
print("Mọi cách đúng với", len(examples + edge), "test")

Mọi cách đúng với 17 test


### 12.2 Ngẫu nhiên "nhiều bẫy" (seed cố định)

Sinh chuỗi từ các mảnh dễ gây nhầm: thực thể thật, thực thể cụt, `&` lẻ, chữ cái, dấu `;`, chữ hoa. Mọi cách phải cho cùng kết quả. Thêm một bản tham chiếu **độc lập**: duyệt mọi vị trí, thử thực thể theo độ dài (viết theo kiểu khác hẳn).

In [20]:
def parse_reference(text):
    res, i = "", 0
    while i < len(text):
        for L in (4, 5, 6, 7):
            piece = text[i:i + L]
            if piece in ENTITIES:
                res += ENTITIES[piece]; i += L
                break
        else:
            res += text[i]; i += 1
    return res

PIECES = list(ENTITIES) + ["&", ";", "amp", "gt", "lt", "quot", "&am", "&fras", "&GT;", "a", "b", " ", "&amp", "l;"]
rng = random.Random(1410)
for _ in range(3000):
    t = "".join(rng.choice(PIECES) for _ in range(rng.randint(1, 12)))
    want = parse_reference(t)
    for name, f in ALL.items():
        assert f(t) == want, (name, t)
print("3000 chuỗi ngẫu nhiên nhiều bẫy: mọi cách khớp bản tham chiếu")

3000 chuỗi ngẫu nhiên nhiều bẫy: mọi cách khớp bản tham chiếu


### 12.3 Ký tự ASCII bất kỳ + giới hạn `n = 10^5`

In [21]:
rng = random.Random(7)
for _ in range(500):
    t = "".join(chr(rng.randrange(256)) if rng.random() < 0.7 else rng.choice(PIECES) for _ in range(rng.randint(1, 40)))
    want = parse_reference(t)
    assert all(f(t) == want for f in ALL.values())
big = "".join(rng.choice(PIECES) for _ in range(30000))[:10**5]
print("độ dài chuỗi lớn:", len(big))
ref = Solution().entityParser(big)
for name, f in ALL.items():
    t0 = time.perf_counter(); r = f(big)
    assert r == ref
    print(f"{name:13s}: {time.perf_counter() - t0:.3f}s")

độ dài chuỗi lớn: 98717


replace      : 0.002s


scan         : 0.008s


scan_submit  : 0.007s
regex        : 0.003s
regex_submit : 0.003s
marker       : 0.008s


submit       : 0.008s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — `replace("&amp;", "&")` chạy **trước**

In [22]:
for t in ["&amp;gt;", "&amp;quot;x&amp;quot;", "&amp;lt;b&amp;gt;"]:
    print(repr(t), "| sai:", repr(parse_replace_naive(t)), "| đúng:", repr(Solution().entityParser(t)))

'&amp;gt;' | sai: '>' | đúng: '&gt;'
'&amp;quot;x&amp;quot;' | sai: '"x"' | đúng: '&quot;x&quot;'
'&amp;lt;b&amp;gt;' | sai: '<b>' | đúng: '&lt;b&gt;'


Giải mã hai lần: `&amp;gt;` → `&gt;` → `>`. **Sửa:** để `&amp;` **cuối cùng**, hoặc quét một lượt.

### Lỗi 2 — Dùng `html.unescape`

In [23]:
def wrong_unescape(text):
    return html.unescape(text)                    # SAI: giải mã quá nhiều loại thực thể

for t in ["&nbsp;", "&#38;", "&ltx", "&copy; 2024", "&amp; is an HTML entity but &ambassador; is not."]:
    print(repr(t), "| sai:", repr(wrong_unescape(t)), "| đúng:", repr(Solution().entityParser(t)))

'&nbsp;' | sai: '\xa0' | đúng: '&nbsp;'
'&#38;' | sai: '&' | đúng: '&#38;'
'&ltx' | sai: '<x' | đúng: '&ltx'
'&copy; 2024' | sai: '© 2024' | đúng: '&copy; 2024'
'&amp; is an HTML entity but &ambassador; is not.' | sai: '& is an HTML entity but &ambassador; is not.' | đúng: '& is an HTML entity but &ambassador; is not.'


`html.unescape` giải mã `&nbsp;`, `&#38;`, `&copy;`, và cả `&lt` **không có dấu `;`** — đề chỉ cho phép 6 thực thể có `;`. **Sửa:** tự cài với bảng 6 thực thể.

### Lỗi 3 — Tìm `;` rồi nhảy cả đoạn khi không khớp

Ý tưởng "từ `&` tìm `;` kế tiếp; nếu đoạn đó không phải thực thể thì chép nguyên đoạn rồi nhảy qua" bỏ sót thực thể **nằm bên trong** đoạn đó.

In [24]:
def wrong_skip_segment(text):
    out, i = [], 0
    while i < len(text):
        if text[i] == "&":
            j = text.find(";", i)
            if j != -1:
                seg = text[i:j + 1]
                out.append(ENTITIES.get(seg, seg))  # SAI: không khớp thì chép cả đoạn và nhảy qua
                i = j + 1
                continue
        out.append(text[i]); i += 1
    return "".join(out)

for t in ["&&gt;", "&x &lt;", "&amp; is an HTML entity but &ambassador; is not."]:
    print(repr(t), "| sai:", repr(wrong_skip_segment(t)), "| đúng:", repr(Solution().entityParser(t)))

'&&gt;' | sai: '&&gt;' | đúng: '&>'
'&x &lt;' | sai: '&x &lt;' | đúng: '&x <'
'&amp; is an HTML entity but &ambassador; is not.' | sai: '& is an HTML entity but &ambassador; is not.' | đúng: '& is an HTML entity but &ambassador; is not.'


`"&x &lt;"`: từ `&` đầu tìm được `;` ở cuối, đoạn `"&x &lt;"` không phải thực thể ⇒ chép nguyên ⇒ bỏ sót `&lt;`. **Sửa:** không khớp thì chỉ chép **một** ký tự `&` và tăng `i` lên 1.

### Lỗi 4 — Cửa sổ tìm `;` quá ngắn (quên `&frasl;` dài 7)

In [25]:
def wrong_window(text):
    out, i = [], 0
    while i < len(text):
        if text[i] == "&":
            j = text.find(";", i, i + 6)            # SAI: cửa sổ 6 ký tự, &frasl; cần 7
            if j != -1 and text[i:j + 1] in ENTITIES:
                out.append(ENTITIES[text[i:j + 1]]); i = j + 1
                continue
        out.append(text[i]); i += 1
    return "".join(out)

for t in ["a&frasl;b", "&quot;&frasl;&quot;"]:
    print(repr(t), "| sai:", repr(wrong_window(t)), "| đúng:", repr(Solution().entityParser(t)))

'a&frasl;b' | sai: 'a&frasl;b' | đúng: 'a/b'
'&quot;&frasl;&quot;' | sai: '"&frasl;"' | đúng: '"/"'


`text.find(";", i, i + 6)` chỉ tìm trong `text[i:i+6]` — tức tối đa 6 ký tự kể cả `&` — nên `;` ở vị trí thứ 7 của `&frasl;` không bao giờ thấy. Cả `&quot;`, `&apos;` dài 6 thì vẫn qua, nên lỗi rất dễ lọt nếu chỉ test hai ví dụ của đề. **Sửa:** cửa sổ `i + 7` (= độ dài thực thể dài nhất), hoặc tính `max(map(len, table))`.

### Lỗi 5 — Regex tham lam `&.*;` hoặc `&.+?;` không giới hạn ký tự

`&.*;` khớp từ `&` **đầu tiên** đến `;` **cuối cùng** — một đoạn khổng lồ không có trong bảng ⇒ không thay gì cả.

In [26]:
def wrong_greedy_regex(text):
    return re.sub(r"&.*;", lambda m: ENTITIES.get(m.group(), m.group()), text)      # SAI: tham lam

def wrong_lazy_regex(text):
    return re.sub(r"&.+?;", lambda m: ENTITIES.get(m.group(), m.group()), text)     # SAI: vẫn nuốt '&' bên trong

for t in ["&lt;b&gt;", "&&gt;", "a &x &lt;"]:
    print(repr(t), "| tham lam:", repr(wrong_greedy_regex(t)), "| lười:", repr(wrong_lazy_regex(t)),
          "| đúng:", repr(Solution().entityParser(t)))

'&lt;b&gt;' | tham lam: '&lt;b&gt;' | lười: '<b>' | đúng: '<b>'
'&&gt;' | tham lam: '&&gt;' | lười: '&&gt;' | đúng: '&>'
'a &x &lt;' | tham lam: 'a &x &lt;' | lười: 'a &x &lt;' | đúng: 'a &x <'


**Sửa:** liệt kê đúng 6 thực thể trong nhóm lựa chọn: `&(?:quot|apos|amp|gt|lt|frasl);`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Thay nhiều mẫu cùng lúc"** ⇒ cảnh giác: `replace` nối tiếp **không** tương đương thay một lượt. Hỏi: kết quả của mẫu này có thể tạo/huỷ mẫu khác không?
- Giải mã (decode) kiểu escape: **mẫu thoát (`&amp;`, `\\`) xử lý cuối** khi decode; **xử lý đầu** khi encode.
- Quét một lượt với con trỏ `i` + `startswith` là cách an toàn nhất, mở rộng dễ.
- Regex với nhóm lựa chọn + hàm thay thế = "thay một lượt" gọn gàng.
- Luôn tự test các bẫy: `&amp;gt;`, `&&gt;`, thiếu `;`, chữ hoa, thực thể dài nhất.

## 15. Biến thể của bài

1. **Encoder ngược** (escape): thay `& " ' > < /` bằng thực thể — phải thay `&` **trước** (Bài 4 mục 17).
2. **Thực thể số** `&#NNN;` / `&#xHH;` ⇒ thêm nhánh đọc số (Bài 5).
3. **Giải mã escape kiểu C/JSON** (`\n`, `\\`, `\"`, `\uXXXX`) — cùng kỹ thuật quét một lượt.
4. **URL decoding** (`%20` → khoảng trắng) — `urllib.parse.unquote`, hoặc tự quét.
5. **Thay nhiều từ khoá cùng lúc** (Aho–Corasick khi số mẫu lớn) — ví dụ LeetCode 833 *Find And Replace in String* (thay theo chỉ số trên chuỗi gốc).
6. **Thêm thực thể sinh ra chữ cái** (ví dụ `&semi;` → `;`) ⇒ cách `replace` nối tiếp càng dễ vỡ; cách quét một lượt vẫn đúng.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. `replace` nối tiếp | 6 lần `replace`, `&amp;` cuối | `O(6n)` | `O(n)` | nhanh khi viết, nhưng phải nhớ thứ tự |
| 2. Quét + `startswith` | con trỏ `i`, thử thực thể tại `&` | `O(n)` | `O(n)` | **mặc định**, an toàn, dễ mở rộng |
| 3. `re.sub` | nhóm lựa chọn + hàm thay | `O(n)` | `O(n)` | code ngắn nhất |
| 4. Đánh dấu `&` | khi gặp `;` kiểm tra từ `&` gần nhất | `O(n)` | `O(n)` | xử lý dạng luồng (stream) từng ký tự |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
`Solution().entityParser("&amp;lt;")` và `Solution().entityParser("&lt;&lt;&gt")` trả về gì?

### Bài 2 (tính tay)
Giải mã `"x &gt;&lt; y &amp;&amp; z &frasl;&quot;"` bằng cách quét tay từ trái sang phải.

### Bài 3 (điền TODO)
Hoàn thiện `parse_todo(text)` theo Cách 2 (chỗ thử thực thể tại `&`).

### Bài 4 (tự viết lại)
Viết `escape(text)` — phép **ngược**: thay `&`, `"`, `'`, `>`, `<`, `/` bằng thực thể sao cho `entityParser(escape(t)) == t` với mọi `t`.

### Bài 5 (mini challenge)
Mở rộng parser hỗ trợ thêm **thực thể số thập phân** `&#N;` với `0 ≤ N ≤ 255` (ví dụ `&#65;` → `A`); các dạng khác giữ nguyên. Viết `parse_numeric(text)`.

In [27]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def parse_todo(text):
    out, i = [], 0
    while i < len(text):
        if text[i] == "&":
            # TODO: thử từng thực thể trong ENTITIES bằng text.startswith(ent, i)
            # TODO: khớp -> ghi ký tự, nhảy len(ent); không khớp -> chép '&', i += 1
            pass
        else:
            out.append(text[i]); i += 1
    return "".join(out)

# Bài 4 — TODO
def escape(text):
    # TODO: thay '&' trước tiên, rồi các ký tự còn lại
    pass

# Bài 5 — TODO
def parse_numeric(text):
    # TODO: như Cách 2; thêm nhánh: text.startswith("&#", i) -> đọc chữ số tới ';' (tối đa 3 chữ số)
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `&amp;` khớp tại 0 ⇒ ghi `&`, nhảy tới `lt;`. (2) Không giải mã lại ⇒ `"&lt;"`. (3) `"&lt;&lt;&gt"` ⇒ hai `&lt;` thành `<<`, `&gt` thiếu `;` giữ nguyên ⇒ `"<<&gt"`.

**Bài 2.** (1) `&gt;` → `>`, `&lt;` → `<`. (2) `&amp;&amp;` → `&&`. (3) `&frasl;` → `/`, `&quot;` → `"` ⇒ `x >< y && z /"`.

**Bài 3.** (1) `for ent, ch in ENTITIES.items(): if text.startswith(ent, i): ...`. (2) Dùng `for … else` để biết không khớp. (3) Kiểm tra với `Solution` trên chuỗi ngẫu nhiên.

**Bài 4.** (1) Nếu thay `"` trước thì `&quot;` sinh ra chứa `&`, sau đó thay `&` sẽ phá nó. (2) Vậy thay `&` → `&amp;` **đầu tiên**. (3) Hoặc quét từng ký tự với bảng ngược — không lo thứ tự.

**Bài 5.** (1) Tại `&#`, tìm `;` trong 3–6 ký tự tiếp theo. (2) Giữa `&#` và `;` phải là 1–3 chữ số, giá trị ≤ 255. (3) Thay bằng `chr(N)`; không hợp lệ thì chép `&`.

## 19. Đáp án tham khảo

In [28]:
# Bài 1, 2
assert Solution().entityParser("&amp;lt;") == "&lt;"
assert Solution().entityParser("&lt;&lt;&gt") == "<<&gt"
assert Solution().entityParser("x &gt;&lt; y &amp;&amp; z &frasl;&quot;") == 'x >< y && z /"'
print("Bài 1, 2: đúng")

Bài 1, 2: đúng


In [29]:
# Bài 3 — đáp án
def parse_todo_sol(text):
    out, i = [], 0
    while i < len(text):
        if text[i] == "&":
            for ent, ch in ENTITIES.items():
                if text.startswith(ent, i):
                    out.append(ch); i += len(ent)
                    break
            else:
                out.append("&"); i += 1
        else:
            out.append(text[i]); i += 1
    return "".join(out)

rng = random.Random(3)
for _ in range(1000):
    t = "".join(rng.choice(PIECES) for _ in range(rng.randint(1, 10)))
    assert parse_todo_sol(t) == Solution().entityParser(t)
print("Bài 3: đúng")

Bài 3: đúng


In [30]:
# Bài 4 — đáp án
REVERSE = {ch: ent for ent, ch in ENTITIES.items()}

def escape_sol(text):
    return "".join(REVERSE.get(ch, ch) for ch in text)      # quét từng ký tự: không lo thứ tự

def escape_replace(text):
    text = text.replace("&", "&amp;")                       # & TRƯỚC TIÊN
    for ch in "\"'></":
        text = text.replace(ch, REVERSE[ch])
    return text

rng = random.Random(4)
alphabet = "ab&;\"'<>/ lgtquotampfrs"
for _ in range(2000):
    t = "".join(rng.choice(alphabet) for _ in range(rng.randint(0, 15)))
    e1, e2 = escape_sol(t), escape_replace(t)
    assert e1 == e2
    assert Solution().entityParser(e1) == t
print(repr(escape_sol('a < b && "x/y"')))
print("Bài 4: đúng")

'a &lt; b &amp;&amp; &quot;x&frasl;y&quot;'
Bài 4: đúng


In [31]:
# Bài 5 — đáp án
def parse_numeric_sol(text):
    out, i, n = [], 0, len(text)
    while i < n:
        if text[i] == "&":
            hit = next((e for e in ENTITIES if text.startswith(e, i)), None)
            if hit:
                out.append(ENTITIES[hit]); i += len(hit)
                continue
            if text.startswith("&#", i):
                j = text.find(";", i + 2, i + 6)             # tối đa 3 chữ số
                digits = text[i + 2:j] if j != -1 else ""
                if 1 <= len(digits) <= 3 and digits.isdigit() and digits.isascii() and int(digits) <= 255:
                    out.append(chr(int(digits))); i = j + 1
                    continue
        out.append(text[i]); i += 1
    return "".join(out)

tests = [("&#65;&#66;C", "ABC"), ("&#38;lt;", "&lt;"), ("&#256;", "&#256;"), ("&#;", "&#;"),
         ("&#0065;", "&#0065;"), ("&#x41;", "&#x41;"), ("&amp;#65;", "&#65;"), ("&lt;&#47;b&gt;", "</b>")]
for t, want in tests:
    assert parse_numeric_sol(t) == want, (t, parse_numeric_sol(t))
rng = random.Random(5)
for _ in range(1000):
    t = "".join(rng.choice(PIECES) for _ in range(rng.randint(1, 10)))
    assert parse_numeric_sol(t) == Solution().entityParser(t)      # không có '&#' thì giống bài gốc
print("Bài 5: đúng")

Bài 5: đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template quét một lượt:**

```python
out, i = [], 0
while i < len(text):
    if text[i] == "&":
        for ent, ch in table.items():
            if text.startswith(ent, i):
                out.append(ch); i += len(ent); break
        else:
            out.append("&"); i += 1
    else:
        out.append(text[i]); i += 1
return "".join(out)
```

**Template regex:** `re.sub(r"&(?:quot|apos|amp|gt|lt|frasl);", lambda m: table[m.group()], text)`

**Nếu dùng `replace`:** `&amp;` **cuối cùng** khi giải mã (đầu tiên khi mã hoá).

**Checklist trước khi submit:**
- [ ] Không giải mã hai lần: `"&amp;gt;"` → `"&gt;"`.
- [ ] `&` lẻ chỉ chép một ký tự rồi xét tiếp: `"&&gt;"` → `"&>"`.
- [ ] Cửa sổ đủ cho `&frasl;` (7 ký tự).
- [ ] Phân biệt hoa thường, bắt buộc có `;`.
- [ ] Không dùng `html.unescape`.
- [ ] Ghép kết quả bằng list + `"".join`.
- [ ] Đúng chữ ký: `entityParser(self, text: str) -> str`.